# Claude Managed Agents로 데이터 분석 agent 만들기

## 소개

모든 팀에는 CSV를 건네받고 "여기서 흥미로운 게 뭐죠?"라는 질문을 받는 사람이 있습니다. 이 cookbook에서는 그 사람을 대신하여 답하는 agent를 만듭니다. CSV를 업로드하면 interactive chart가 포함된 서술형 HTML 보고서를 받습니다.

stateful하고 tool을 사용하는 agent를 위한 Anthropic의 hosted runtime인 [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)에서 실행합니다. 네 가지 핵심 개념을 기반으로 합니다.

- **Agent**: model, system prompt, tool, MCP server, skill
- **Environment**: 설정된 container template(package, 네트워크 접근)
- **Session**: environment 안에서 특정 작업을 수행하고 출력을 생성하는 실행 중인 agent instance
- **Events**: 애플리케이션과 agent 사이에 교환되는 message(user turn, tool 결과, 상태 update)

agent와 environment를 결합하면 세션이 됩니다. 여기에 데이터를 resource로 연결하고 event를 보내며 stream을 읽어 진행합니다.

Anthropic이 sandbox, tool 실행, context 관리를 대신 처리합니다. agent 루프와 deployment를 완전히 제어해야 한다면 대신 [Claude Agent SDK](https://platform.claude.com/docs/en/api/agent-sdk/overview)를 사용해 보세요.

### 배울 내용

이 cookbook을 마치면 다음을 할 수 있습니다.

- 데이터 분석을 위한 재사용 가능한 environment와 agent 설정하기
- agent에 작업할 dataset 제공하기
- 실행 중 agent의 진행 상황 관찰하기
- 보고서와 생성한 다른 파일 다운로드하기

### 사전 요구 사항

- Python 3.11+
- [Console](https://platform.claude.com/settings/keys)에서 발급받아 `ANTHROPIC_API_KEY`로 설정한 Anthropic API key

dependency 설치:


In [10]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [11]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. environment 생성하기

**environment**는 재사용 가능한 container spec입니다. 여기에 `pandas`와 `plotly`를 선언하면 모든 세션이 미리 설치된 상태로 시작하므로 agent는 먼저 `pip install`을 실행하는 대신 즉시 분석을 시작할 수 있습니다.

여기서는 agent가 CDN에서 plotly를 불러올 수 있도록 networking을 `unrestricted`로 설정합니다. 하지만 인터넷 어디든 접근할 수 있게 되므로 production에서는 대신 [host allowlist](https://platform.claude.com/docs/en/managed-agents/environments)를 사용하세요.


In [12]:
env = client.beta.environments.create(
    name="cookbook-data-analyst-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": ["pandas", "plotly"],
        },
    },
)

## 2. agent 생성하기

**agent**는 model을 system prompt 및 tool 집합과 연결합니다. 출력 품질의 대부분은 system prompt에서 결정됩니다. 이 prompt는 서술 구조, 구체적인 수치로 뒷받침된 발견, 여러 plotly chart를 하나의 HTML 파일에 embedding하는 적절한 패턴을 강조합니다.

[`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools)은 `bash`, `read`, `write`, `edit`, `glob`, `grep`, `web_fetch`, `web_search`의 여덟 tool을 제공합니다. 여기서는 모두 `always_allow`로 실행하되 분석이 offline이므로 두 web tool은 비활성화합니다.


In [13]:
ANALYST_SYSTEM_PROMPT = """\
You are a senior data analyst producing a publication-quality report.

## Style
- Professional and precise. Let the data speak with concrete numbers.
- Short paragraphs (2-3 sentences) between charts.
- Lead with the most actionable finding.

## Execution
- Write .py scripts and run them with `python3 script.py`.
- Sample large tables (`nrows=` / `.sample()`) instead of loading everything.
- Sanity-check key metrics before building narrative around them.

## Charts
- Build each chart as its own `go.Figure()`, embed with
  `fig.to_html(include_plotlyjs=False, full_html=False)`, and load plotly
  from the CDN once in <head>.
- Always set `marker_color` and `template='simple_white'`.

## Output
Write a single self-contained `report.html` to /mnt/session/outputs/
with inline CSS, 3+ embedded plotly charts, and a closing section of
actionable recommendations. Confirm "Saved: report.html" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-data-analyst",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/data-analyst-agent"},
    model=MODEL,
    system=ANALYST_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            # default_config는 집합의 모든 tool에 적용합니다.
            # configs 항목은 특정 tool을 덮어씁니다.
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
)

## 3. dataset 업로드하기

포함된 예제 CSV는 50행이므로 몇 분 안에 분석이 완료됩니다. 여기서 아무 CSV(또는 CSV의 zip)로 바꿔도 나머지 흐름은 동일합니다.


In [14]:
DATA_PATH = Path("example_data/data_analyst_agent/sales_data.csv")

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/csv"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

Uploaded sales_data.csv (2833 bytes) as file_011CZqKKomgL45BLo2J8K5X9


## 4. 세션 생성하고 작업 보내기

**session**은 agent를 environment 및 mount된 파일과 연결합니다. `{"type": "agent", "id": ..., "version": ...}`를 전달하면 위에서 생성한 version이 지정된 agent를 재사용합니다. `resources`는 업로드된 파일을 container 내부의 지정된 절대 경로에 mount합니다.

세션 생성 후 작업을 담은 `user.message` event를 보내세요. agent가 즉시 작업을 시작합니다.


In [15]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}],
    title="Sales analysis",
)

ANALYSIS_PROMPT = f"""\
Analyze the e-commerce orders in {MOUNT_PATH}.

Columns: order_id, customer_id, product, category, price, quantity,
order_date, region.

Focus on revenue by category and region, repeat-customer behavior, and
one surprising pattern. Produce report.html per your system instructions.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": ANALYSIS_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

Session sesn_011CZqKKqyPRph9J5khKLn6u running


## 5. 실행 streaming하기

[Console](https://platform.claude.com/)의 **Sessions**에서 세션을 열어 모든 event, tool 호출, token 수를 실시간으로 확인하세요.

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/data_analyst_agent/console_session.png" alt="Console의 세션 trace" width="700" />

아래 helper는 같은 event stream을 따라 읽으며 `agent.message` text와 `agent.tool_use` 호출을 도착하는 대로 출력하고 `session.status_idle`에서 반환합니다.


In [ ]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

## 6. 보고서 가져오기

agent가 `/mnt/session/outputs/`에 쓰는 모든 것은 보존되며 `scope_id=<session_id>`를 사용하는 Files API로 제공됩니다. container의 다른 곳에 쓴 파일은 보존되지 않습니다.

[Files API](https://platform.claude.com/docs/en/api/beta/files/list)는 별도 beta 기능이므로 여기서 `scope_id`를 사용하려면 Managed Agents beta header도 전달해야 합니다.


In [17]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

# 목록에는 mount된 입력 CSV도 포함됩니다. 보고서를 골라냅니다.
report = next((f for f in outputs.data if f.filename == "report.html"), None)
if report is None:
    raise RuntimeError(f"report.html not found. Files: {[f.filename for f in outputs.data]}")
content = client.beta.files.download(report.id)
Path("report.html").write_bytes(content.read())
print("Downloaded report.html")

report.html 53728
sales_data.csv 2833
Downloaded report.html


## 7. 정리와 다음 단계

agent와 environment는 한 번 생성하여 여러 실행에서 재사용하고, 대화마다 새 세션을 생성합니다. 이제 보고서가 있으므로 이 세션을 archive하여 container를 해제하세요. 아래 코드는 agent 및 environment ID를 `.env`에 저장하여 [`slack_data_bot.ipynb`](slack_data_bot.ipynb)이 이를 사용해 새 세션을 시작할 수 있도록 합니다.

> **경고:** 다음 셀을 실행하기 전에 `.env`가 `.gitignore`에 포함되어 있는지 확인하세요. 절대로 commit하지 마세요.


In [18]:
client.beta.sessions.archive(session.id)

set_key(".env", "ANALYST_ENV_ID", env.id)
set_key(".env", "ANALYST_AGENT_ID", agent.id)
set_key(".env", "ANALYST_AGENT_VERSION", str(agent.version))
print("Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env")

Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env


데이터 분석 agent를 처음부터 끝까지 만들고 실행했습니다. 재사용 가능한 environment와 agent, CSV를 mount한 세션, 실시간 event stream, 다운로드한 HTML 보고서입니다.

이제 다음을 할 수 있습니다.

- 다운로드한 `report.html`을 열어 서술과 chart를 확인하세요.
- [Console](https://platform.claude.com/)에서 세션을 열어 token 사용량과 전체 event log를 살펴보세요.
- [`slack_data_bot.ipynb`](slack_data_bot.ipynb)으로 이어가며 이 agent를 Slack에서 진행하세요.
